# 05 — Lending Club feature engineering

Transform **lendingclub_pruned.csv** into an ML-oriented table for risk classification and investor-style signals. **No model training** — engineering and labeling only.

**Input:** `datasets/processed/` (auto-resolved pruned CSV)  
**Output:** `datasets/processed/lendingclub_ml_ready.csv`

**Note:** `risk_category` is a **synthetic MVP label** derived from observable credit metrics (not LC realized outcomes).


## 1. Load data


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT: Path
DATA_PATH: Path
df: pd.DataFrame


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for directory in (cwd, *cwd.parents):
        if (directory / "backend").is_dir() and (directory / "ml").is_dir():
            return directory
    return cwd


def resolve_pruned_csv(processed_dir: Path) -> Path:
    if not processed_dir.is_dir():
        raise FileNotFoundError(
            f"Processed datasets directory not found: {processed_dir}. "
            "Run 04_feature_pruning.ipynb first or create the folder."
        )
    canonical = processed_dir / "lendingclub_pruned.csv"
    if canonical.is_file():
        return canonical.resolve()
    candidates = sorted(
        p.resolve() for p in processed_dir.glob("*.csv") if "pruned" in p.name.lower()
    )
    if not candidates:
        listing = sorted(p.name for p in processed_dir.iterdir()) if processed_dir.exists() else []
        raise FileNotFoundError(
            f"No pruned Lending Club CSV in {processed_dir.resolve()}. "
            f"Expected 'lendingclub_pruned.csv' or a filename containing 'pruned'. "
            f"Directory contents: {listing or '(empty)'}"
        )
    return candidates[0]


def fmt_mem_bytes(n: int) -> str:
    return f"{n / (1024 ** 2):.2f} MiB ({n:,} bytes)"


REPO_ROOT = find_repo_root()
PROC_DIR = REPO_ROOT / "datasets" / "processed"
DATA_PATH = resolve_pruned_csv(PROC_DIR)

print(f"Resolved CSV: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, low_memory=False)
mem_load = int(df.memory_usage(deep=True).sum())
print(f"Shape: {df.shape}")
print(f"Memory (deep estimate): {fmt_mem_bytes(mem_load)}")


## 2. Basic inspection


In [ ]:
df.info()
df.head(5)


## 3. Clean text / mixed columns


In [ ]:
def clean_percent_series(s: pd.Series) -> pd.Series:
    out = s.astype(str).str.replace("%", "", regex=False).str.strip()
    out = out.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return pd.to_numeric(out, errors="coerce")


def parse_term_to_months(val) -> float:
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+)", str(val))
    return float(m.group(1)) if m else np.nan


def parse_emp_length_years(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in {"", "nan", "n/a", "na"}:
        return np.nan
    if "<" in s or "less than" in s:
        return 0.0
    if "10+" in s or "10 +" in s:
        return 10.0
    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan


if "term" in df.columns:
    df["term"] = df["term"].map(parse_term_to_months).astype("float64")
    print("Cleaned: term -> numeric months")

if "emp_length" in df.columns:
    df["emp_length"] = df["emp_length"].map(parse_emp_length_years)
    print("Cleaned: emp_length -> numeric years (NaN for n/a)")

for col in ("int_rate", "revol_util"):
    if col in df.columns and df[col].dtype == object:
        df[col] = clean_percent_series(df[col])
        print(f"Cleaned: {col} -> numeric (stripped %)")


## 4. Date feature engineering


In [ ]:
if {"issue_d", "earliest_cr_line"}.issubset(df.columns):
    issue_dt = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    early_dt = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
    df["credit_history_years"] = (issue_dt - early_dt).dt.days / 365.25
    df = df.drop(columns=["issue_d", "earliest_cr_line"])
    print("Created: credit_history_years; dropped issue_d, earliest_cr_line")
else:
    missing = {"issue_d", "earliest_cr_line"} - set(df.columns)
    print(f"Skipped date features (missing columns): {sorted(missing)}")


## 5. Drop low-value / noisy columns


In [ ]:
DROP_LOW_VALUE = [
    "addr_state",
    "policy_code",
    "pymnt_plan",
    "initial_list_status",
    "hardship_flag",
    "disbursement_method",
]
present = [c for c in DROP_LOW_VALUE if c in df.columns]
if present:
    df = df.drop(columns=present)
    print(f"Dropped ({len(present)}): {present}")
else:
    print("No low-value columns from list were present.")


## 6. Feature engineering


In [ ]:
def minmax_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = float(s.quantile(0.01)), float(s.quantile(0.99))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return pd.Series(0.0, index=s.index, dtype="float64")
    clipped = s.clip(lo, hi)
    return (clipped - lo) / (hi - lo)


if {"fico_range_low", "fico_range_high"}.issubset(df.columns):
    df["fico_avg"] = (pd.to_numeric(df["fico_range_low"], errors="coerce") + pd.to_numeric(df["fico_range_high"], errors="coerce")) / 2
    print("Created: fico_avg")

if {"annual_inc", "loan_amnt"}.issubset(df.columns):
    loan_amnt = pd.to_numeric(df["loan_amnt"], errors="coerce").replace(0, np.nan)
    df["income_to_loan_ratio"] = pd.to_numeric(df["annual_inc"], errors="coerce") / loan_amnt
    print("Created: income_to_loan_ratio")

if {"installment", "annual_inc"}.issubset(df.columns):
    inc = pd.to_numeric(df["annual_inc"], errors="coerce").replace(0, np.nan)
    df["installment_to_income_ratio"] = (pd.to_numeric(df["installment"], errors="coerce") * 12) / inc
    print("Created: installment_to_income_ratio")

parts = []
weights = []
for col, w in (("dti", 0.30), ("revol_util", 0.30), ("delinq_2yrs", 0.20), ("inq_last_6mths", 0.20)):
    if col in df.columns:
        parts.append(minmax_series(df[col]) * w)
        weights.append(w)
if parts:
    ssum = sum(parts)
    wsum = sum(weights)
    df["debt_pressure_score"] = ssum / wsum if wsum else ssum
    print("Created: debt_pressure_score (weighted min-max components)")

if "emp_length" in df.columns:
    el = df["emp_length"]

    def _stab(y):
        if pd.isna(y):
            return np.nan
        if y <= 1:
            return 0.0
        if y <= 5:
            return 1.0
        return 2.0

    df["employment_stability_score"] = el.map(_stab)
    print("Created: employment_stability_score (0=low,1=med,2=high tenure)")

risk_parts = []
risk_ws = []
if "fico_avg" in df.columns:
    f = pd.to_numeric(df["fico_avg"], errors="coerce")
    inv = (850.0 - f.clip(300, 850)) / (850.0 - 300.0)
    risk_parts.append(inv * 0.35)
    risk_ws.append(0.35)
for col, w in (("dti", 0.20), ("revol_util", 0.20), ("delinq_2yrs", 0.15), ("pub_rec", 0.10)):
    if col in df.columns:
        risk_parts.append(minmax_series(df[col]) * w)
        risk_ws.append(w)
if risk_parts:
    rsum = sum(risk_parts)
    rw = sum(risk_ws)
    df["credit_risk_score"] = rsum / rw if rw else rsum
    print("Created: credit_risk_score (higher = riskier, MVP blend)")

df = df.replace([np.inf, -np.inf], np.nan)
print("Replaced inf values with NaN where produced by ratios.")


## 7. Encode categorical features


In [ ]:
GRADE_ORDER = list("ABCDEFG")
CATS_ONEHOT = ["home_ownership", "verification_status", "purpose", "application_type"]

encoded_dummy_cols: list[str] = []

for _col in CATS_ONEHOT:
    if _col in df.columns:
        _mode = df[_col].mode()
        df[_col] = df[_col].fillna(_mode.iloc[0] if len(_mode) else "unknown")
print("Mode-imputed categorical columns prior to one-hot (if present).")


def encode_grade(val) -> float:
    if pd.isna(val):
        return np.nan
    ch = str(val).strip().upper()[:1]
    if ch not in GRADE_ORDER:
        return np.nan
    return float(GRADE_ORDER.index(ch) + 1)


def encode_sub_grade(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if len(s) < 2:
        return np.nan
    letter, digit = s[0], s[1]
    if letter not in GRADE_ORDER or not digit.isdigit():
        return np.nan
    return float(GRADE_ORDER.index(letter) * 5 + int(digit))


if "grade" in df.columns:
    df["grade_encoded"] = df["grade"].map(encode_grade)
    df = df.drop(columns=["grade"])
    print("Encoded: grade -> grade_encoded (ordinal A=1..G=7)")

if "sub_grade" in df.columns:
    df["sub_grade_encoded"] = df["sub_grade"].map(encode_sub_grade)
    df = df.drop(columns=["sub_grade"])
    print("Encoded: sub_grade -> sub_grade_encoded (ordinal A1..G5)")

for col in CATS_ONEHOT:
    if col not in df.columns:
        print(f"Skip one-hot (missing): {col}")
        continue
    dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False, dtype=np.int8)
    encoded_dummy_cols.extend(dummies.columns.astype(str).tolist())
    df = pd.concat([df.drop(columns=[col]), dummies], axis=1)
    print(f"One-hot encoded: {col} (+{dummies.shape[1]} columns)")

print(f"Total dummy columns: {len(encoded_dummy_cols)}")


## 8. Handle missing values


In [ ]:
from IPython.display import display


def missing_table(frame: pd.DataFrame) -> pd.DataFrame:
    m = frame.isna().sum()
    p = (m / max(len(frame), 1) * 100).round(4)
    return (
        pd.DataFrame({"column": m.index, "missing_count": m.values, "missing_pct": p.values})
        .sort_values("missing_count", ascending=False)
        .reset_index(drop=True)
    )


print("=== Missing summary BEFORE imputation ===")
before = missing_table(df)
display(before)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

for c in num_cols:
    med = df[c].median()
    if pd.isna(med):
        med = 0.0
    df[c] = df[c].fillna(med)

for c in obj_cols:
    mode = df[c].mode()
    fill = mode.iloc[0] if len(mode) else ""
    df[c] = df[c].fillna(fill)

print("=== Missing summary AFTER imputation ===")
after = missing_table(df)
display(after)
print(f"Total missing cells after: {int(df.isna().sum().sum())}")


## 9. Feature selection (~25–40 features)


In [ ]:
CORE_NUMERIC = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_avg",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "credit_history_years",
    "income_to_loan_ratio",
    "installment_to_income_ratio",
    "debt_pressure_score",
    "employment_stability_score",
    "credit_risk_score",
    "grade_encoded",
    "sub_grade_encoded",
]

MAX_FEATURES = 40
MIN_FEATURES = 25

selected: list[str] = []
for c in CORE_NUMERIC:
    if c in df.columns and c not in selected:
        selected.append(c)

for c in sorted(encoded_dummy_cols):
    if c in df.columns and c not in selected:
        selected.append(c)
    if len(selected) >= MAX_FEATURES:
        break

if len(selected) < MIN_FEATURES:
    for c in sorted(encoded_dummy_cols):
        if c in df.columns and c not in selected:
            selected.append(c)
        if len(selected) >= MIN_FEATURES:
            break

selected = [c for c in selected if c in df.columns]
print(f"Selected feature count: {len(selected)} (target band {MIN_FEATURES}–{MAX_FEATURES})")
print(selected)


## 10. Create target label (`risk_category`)


In [ ]:
needed = {"fico_avg", "dti", "delinq_2yrs"}
if not needed.issubset(df.columns):
    raise KeyError(f"Cannot build risk_category; missing: {sorted(needed - set(df.columns))}")

fico = pd.to_numeric(df["fico_avg"], errors="coerce")
dti = pd.to_numeric(df["dti"], errors="coerce")
delinq = pd.to_numeric(df["delinq_2yrs"], errors="coerce").fillna(0)

high = (fico < 640) | (dti > 40) | (delinq >= 2)
low = (fico >= 720) & (dti <= 28) & (delinq <= 0)
# 0 = low, 1 = medium, 2 = high (HIGH evaluated first)
df["risk_category"] = np.select([high, low], [2, 0], default=1).astype(np.int8)

print("risk_category distribution:")
print(df["risk_category"].value_counts().sort_index())
print(df["risk_category"].value_counts(normalize=True).sort_index().round(4))


## 11. Final validation


In [ ]:
out = df[selected + ["risk_category"]].copy()

print("=== Final validation ===")
print(f"Shape: {out.shape}")
print("dtypes:")
print(out.dtypes)
print("\nFeature names:")
print(list(out.columns))
print(f"\nMissing values (should be 0): {int(out.isna().sum().sum())}")
print("\nClass distribution:")
print(out["risk_category"].value_counts().sort_index())
mem_out = int(out.memory_usage(deep=True).sum())
print(f"\nMemory (deep estimate): {fmt_mem_bytes(mem_out)}")


## 12. Save ML-ready dataset


In [ ]:
OUTPUT_PATH = REPO_ROOT / "datasets" / "processed" / "lendingclub_ml_ready.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote: {OUTPUT_PATH.resolve()}  |  shape: {out.shape}")
